In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))

2025-12-23 03:22:41.889235: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766460162.117555      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766460162.189050      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766460162.744101      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766460162.744139      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766460162.744142      55 computation_placer.cc:177] computation placer alr

2.19.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [3]:
tf.test.is_built_with_cuda()

True

In [4]:
tf.config.list_physical_devices("GPU")

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'),
 PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]

In [5]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
cpus = tf.config.list_physical_devices("CPU")

print("GPUs detected:", len(gpus))
for i, g in enumerate(gpus):
    print(f" GPU {i}:", g)

print("\nCPUs detected:", len(cpus))
for c in cpus:
    print(" ", c)

GPUs detected: 2
 GPU 0: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
 GPU 1: PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')

CPUs detected: 1
  PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')


In [6]:
import os
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

# Detect GPUs
gpus = tf.config.list_physical_devices("GPU")
print("GPUs detected:", len(gpus))
for i, g in enumerate(gpus):
    print(f" GPU {i}:", g)

# Detect CPU threads
print("CPU threads available:", os.cpu_count())

# Use all CPU threads
tf.config.threading.set_intra_op_parallelism_threads(0)
tf.config.threading.set_inter_op_parallelism_threads(0)

# Avoid TF grabbing all GPU memory
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print("Environment ready")

TensorFlow version: 2.19.0
GPUs detected: 2
 GPU 0: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
 GPU 1: PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')
CPU threads available: 4
Environment ready


In [3]:
# Dataset + input pipeline
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import random

SEED = 12345
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0
y_train = y_train.squeeze()
y_test  = y_test.squeeze()

VALIDATION_SPLIT = 0.1
n = x_train.shape[0]
idx = np.random.permutation(n)
val_n = int(n * VALIDATION_SPLIT)

x_val, y_val = x_train[idx[:val_n]], y_train[idx[:val_n]]
x_train, y_train = x_train[idx[val_n:]], y_train[idx[val_n:]]

BATCH_SIZE = 128

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(20000, seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("Dataset ready")

Dataset ready


In [8]:
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow as tf

strategy = tf.distribute.MirroredStrategy()
print("GPUs in sync:", strategy.num_replicas_in_sync)

with strategy.scope():
    model_dp = keras.Sequential([
        layers.Input(shape=(32, 32, 3)),

        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),

        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])

    model_dp.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[keras.metrics.SparseCategoricalAccuracy()]
    )

model_dp.summary()

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
GPUs in sync: 2


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     1,048,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,116,970 (4.26 MB)

 Trainable params: 1,116,970 (4.26 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
model_dp.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20
)

dp_loss, dp_acc = model_dp.evaluate(x_test, y_test, verbose=0)
print("Data Parallel Test Accuracy:", dp_acc)

INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Redu

I0000 00:00:1766459893.963676     167 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1766459893.963694     164 cuda_dnn.cc:529] Loaded cuDNN version 91002


352/352 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - loss: 1.7499 - sparse_categorical_accuracy: 0.3627 - val_loss: 1.2047 - val_sparse_categorical_accuracy: 0.5738
Epoch 2/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - loss: 1.1145 - sparse_categorical_accuracy: 0.6061 - val_loss: 0.9754 - val_sparse_categorical_accuracy: 0.6582
Epoch 3/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - loss: 0.8657 - sparse_categorical_accuracy: 0.6990 - val_loss: 0.8694 - val_sparse_categorical_accuracy: 0.7026
Epoch 4/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - loss: 0.7206 - sparse_categorical_accuracy: 0.7495 - val_loss: 0.8351 - val_sparse_categorical_accuracy: 0.7130
Epoch 5/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - loss: 0.6003 - sparse_categorical_accuracy: 0.7906 - val_loss: 0.8121 - val_sparse_categorical_accuracy: 0.7240
Epoch 6/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - loss: 0.4899 - sparse_categorical_accuracy: 0.8267 - val_loss: 0.8167 - val_sparse_categorical_accuracy: 0.7272
Epoch 7/

In [4]:
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow as tf


def resnet_block(x, filters, stride=1):
    shortcut = x

    x = layers.Conv2D(filters, 3, strides=stride, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv2D(filters, 3, strides=1, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)

    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.Activation("relu")(x)
    return x



def build_resnet18_cifar():
    inputs = keras.Input(shape=(32, 32, 3))

    x = layers.Conv2D(64, 3, padding="same", use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = resnet_block(x, 64)
    x = resnet_block(x, 64)

    x = resnet_block(x, 128, stride=2)
    x = resnet_block(x, 128)

    x = resnet_block(x, 256, stride=2)
    x = resnet_block(x, 256)

    x = resnet_block(x, 512, stride=2)
    x = resnet_block(x, 512)

    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(10, activation="softmax")(x)

    return keras.Model(inputs, outputs)


strategy = tf.distribute.MirroredStrategy()
print("GPUs in sync:", strategy.num_replicas_in_sync)

with strategy.scope():
    model_resnet_dp = build_resnet18_cifar()
    model_resnet_dp.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[keras.metrics.SparseCategoricalAccuracy()]
    )

model_resnet_dp.summary()

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
GPUs in sync: 2


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 32, 32,    │      1,728 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 32, 32,    │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 32, 32,    │     36,864 │ activation[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 32, 32,    │     36,864 │ activation_1[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 32, 32,    │          0 │ add[0][0]         │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 32, 32,    │     36,864 │ activation_2[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 32, 32,    │     36,864 │ activation_3[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 32, 32,    │          0 │ batch_normalizat

 Total params: 11,183,562 (42.66 MB)

 Trainable params: 11,173,962 (42.63 MB)

 Non-trainable params: 9,600 (37.50 KB)

In [5]:
model_resnet_dp.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Redu

I0000 00:00:1766460237.759912     126 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1766460237.760308     124 cuda_dnn.cc:529] Loaded cuDNN version 91002


352/352 ━━━━━━━━━━━━━━━━━━━━ 54s 112ms/step - loss: 1.6098 - sparse_categorical_accuracy: 0.4423 - val_loss: 2.2986 - val_sparse_categorical_accuracy: 0.3332
Epoch 2/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 38s 106ms/step - loss: 0.8269 - sparse_categorical_accuracy: 0.7076 - val_loss: 1.5912 - val_sparse_categorical_accuracy: 0.5498
Epoch 3/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 38s 107ms/step - loss: 0.5856 - sparse_categorical_accuracy: 0.7971 - val_loss: 0.7920 - val_sparse_categorical_accuracy: 0.7384
Epoch 4/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 39s 108ms/step - loss: 0.4374 - sparse_categorical_accuracy: 0.8491 - val_loss: 1.5053 - val_sparse_categorical_accuracy: 0.6216
Epoch 5/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 39s 108ms/step - loss: 0.3351 - sparse_categorical_accuracy: 0.8834 - val_loss: 0.6905 - val_sparse_categorical_accuracy: 0.7844
Epoch 6/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 39s 108ms/step - loss: 0.2470 - sparse_categorical_accuracy: 0.9134 - val_loss: 0.7093 - val_sparse_categorical_accuracy: 0.78

In [1]:

import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import random

SEED = 12345
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)


tf.config.threading.set_intra_op_parallelism_threads(0)
tf.config.threading.set_inter_op_parallelism_threads(0)


gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print("TensorFlow:", tf.__version__)
print("GPUs:", gpus)
print("CPU threads:", os.cpu_count())


(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0
y_train = y_train.squeeze()
y_test  = y_test.squeeze()


VALIDATION_SPLIT = 0.1
n = x_train.shape[0]
idx = np.random.permutation(n)
val_n = int(n * VALIDATION_SPLIT)

x_val, y_val = x_train[idx[:val_n]], y_train[idx[:val_n]]
x_train, y_train = x_train[idx[val_n:]], y_train[idx[val_n:]]

BATCH_SIZE = 128

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(20000, seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("Dataset ready")



def resnet_block(x, filters, stride=1):
    shortcut = x

    x = layers.Conv2D(
        filters, 3, strides=stride, padding="same",
        use_bias=False,
        kernel_regularizer=keras.regularizers.l2(1e-4)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv2D(
        filters, 3, padding="same",
        use_bias=False,
        kernel_regularizer=keras.regularizers.l2(1e-4)
    )(x)
    x = layers.BatchNormalization()(x)

    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(
            filters, 1, strides=stride,
            use_bias=False,
            kernel_regularizer=keras.regularizers.l2(1e-4)
        )(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.Activation("relu")(x)
    return x


def build_resnet18_cifar():
    inputs = keras.Input(shape=(32, 32, 3))

    x = data_augmentation(inputs)

    x = layers.Conv2D(
        64, 3, padding="same", use_bias=False,
        kernel_regularizer=keras.regularizers.l2(1e-4)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    # Stage 1
    x = resnet_block(x, 64)
    x = resnet_block(x, 64)

    # Stage 2
    x = resnet_block(x, 128, stride=2)
    x = resnet_block(x, 128)

    # Stage 3
    x = resnet_block(x, 256, stride=2)
    x = resnet_block(x, 256)

    # Stage 4
    x = resnet_block(x, 512, stride=2)
    x = resnet_block(x, 512)

    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(10, activation="softmax")(x)

    return keras.Model(inputs, outputs)

# ============================================================
# Data Parallelism (2 GPUs)
# ============================================================
strategy = tf.distribute.MirroredStrategy()
print("GPUs in sync:", strategy.num_replicas_in_sync)

with strategy.scope():
    model = build_resnet18_cifar()

    lr_schedule = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.1,
        decay_steps=50000
    )

    optimizer = keras.optimizers.SGD(
        learning_rate=lr_schedule,
        momentum=0.9
    )

    model.compile(
        optimizer=optimizer,
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[keras.metrics.SparseCategoricalAccuracy()]
    )

model.summary()


history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10   # increase to 20 for ~90%
)


test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print("Final Test Accuracy:", test_acc)

2025-12-23 03:31:23.952309: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766460684.176148      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766460684.253113      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766460684.810821      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766460684.810861      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766460684.810863      55 computation_placer.cc:177] computation placer alr

TensorFlow: 2.19.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
CPU threads: 4
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


I0000 00:00:1766460706.445972      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1766460706.449916      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Dataset ready
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
GPUs in sync: 2


Model: "functional_1"

...

 Trainable params: 11,173,962 (42.63 MB)

 Non-trainable params: 9,600 (37.50 KB)

INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Redu

I0000 00:00:1766460722.455181     124 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1766460723.357789     122 cuda_dnn.cc:529] Loaded cuDNN version 91002


352/352 ━━━━━━━━━━━━━━━━━━━━ 54s 112ms/step - loss: 2.9331 - sparse_categorical_accuracy: 0.2714 - val_loss: 2.2015 - val_sparse_categorical_accuracy: 0.3788
Epoch 2/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 38s 105ms/step - loss: 1.8780 - sparse_categorical_accuracy: 0.4820 - val_loss: 1.8037 - val_sparse_categorical_accuracy: 0.5302
Epoch 3/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 39s 108ms/step - loss: 1.5443 - sparse_categorical_accuracy: 0.6008 - val_loss: 1.6331 - val_sparse_categorical_accuracy: 0.5766
Epoch 4/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 40s 111ms/step - loss: 1.2825 - sparse_categorical_accuracy: 0.6889 - val_loss: 1.4788 - val_sparse_categorical_accuracy: 0.6378
Epoch 5/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 39s 110ms/step - loss: 1.0930 - sparse_categorical_accuracy: 0.7518 - val_loss: 1.1649 - val_sparse_categorical_accuracy: 0.7246
Epoch 6/10
352/352 ━━━━━━━━━━━━━━━━━━━━ 39s 110ms/step - loss: 0.9455 - sparse_categorical_accuracy: 0.7958 - val_loss: 1.2310 - val_sparse_categorical_accuracy: 0.71

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import time
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import random

PROFILE = []

def tick(name, t0):
    PROFILE.append((name, time.perf_counter() - t0))

def now():
    return time.perf_counter()

t0 = now()

SEED = 12345
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

tf.config.threading.set_intra_op_parallelism_threads(0)
tf.config.threading.set_inter_op_parallelism_threads(0)

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

tick("environment_setup", t0)

print("TensorFlow:", tf.__version__)
print("GPUs:", gpus)
print("CPU threads:", os.cpu_count())

t0 = now()

(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
y_train = y_train.squeeze()
y_test = y_test.squeeze()

tick("dataset_load_and_preprocess", t0)

t0 = now()

VALIDATION_SPLIT = 0.1
n = x_train.shape[0]
idx = np.random.permutation(n)
val_n = int(n * VALIDATION_SPLIT)

x_val, y_val = x_train[idx[:val_n]], y_train[idx[:val_n]]
x_train, y_train = x_train[idx[val_n:]], y_train[idx[val_n:]]

tick("train_val_split", t0)

t0 = now()

BATCH_SIZE = 128

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(20000, seed=SEED)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

for _ in train_ds.take(5):
    pass

tick("tfdata_pipeline_build", t0)

print("Dataset ready")

def resnet_block(x, filters, stride=1):
    shortcut = x
    x = layers.Conv2D(filters, 3, strides=stride, padding="same", use_bias=False,
                      kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False,
                      kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, use_bias=False,
                                 kernel_regularizer=keras.regularizers.l2(1e-4))(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    x = layers.Add()([x, shortcut])
    x = layers.Activation("relu")(x)
    return x

def build_resnet18_cifar():
    inputs = keras.Input(shape=(32, 32, 3))
    x = layers.Conv2D(64, 3, padding="same", use_bias=False,
                      kernel_regularizer=keras.regularizers.l2(1e-4))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = resnet_block(x, 64)
    x = resnet_block(x, 64)
    x = resnet_block(x, 128, stride=2)
    x = resnet_block(x, 128)
    x = resnet_block(x, 256, stride=2)
    x = resnet_block(x, 256)
    x = resnet_block(x, 512, stride=2)
    x = resnet_block(x, 512)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(10, activation="softmax")(x)
    return keras.Model(inputs, outputs)

t0 = now()

strategy = tf.distribute.MirroredStrategy()
print("GPUs in sync:", strategy.num_replicas_in_sync)

with strategy.scope():
    model = build_resnet18_cifar()
    lr_schedule = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.1,
        decay_steps=50000
    )
    optimizer = keras.optimizers.SGD(
        learning_rate=lr_schedule,
        momentum=0.9
    )
    model.compile(
        optimizer=optimizer,
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[keras.metrics.SparseCategoricalAccuracy()]
    )

tick("model_build_and_compile", t0)

model.summary()

t0 = now()
model.train_on_batch(x_train[:128], y_train[:128])
tick("single_train_step", t0)

EPOCHS = 100
steps_per_epoch = x_train.shape[0] // BATCH_SIZE
validation_steps = x_val.shape[0] // BATCH_SIZE

t0 = now()
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    verbose=1
)
tick("full_training_time", t0)

t0 = now()
model.evaluate(val_ds, verbose=0)
tick("validation_time", t0)

t0 = now()
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
tick("test_inference_time", t0)

print("Final Test Accuracy:", test_acc)

print("PROFILING SUMMARY ==========")
for k, v in PROFILE:
    print(f"{k:30s}: {v:.4f} sec")

2025-12-23 04:03:19.546544: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766462599.738501      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766462599.788696      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766462600.263339      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766462600.263381      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766462600.263384      55 computation_placer.cc:177] computation placer alr

TensorFlow: 2.19.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
CPU threads: 4
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


I0000 00:00:1766462620.398631      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1766462620.399318      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Dataset ready
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
GPUs in sync: 2


Model: "functional"

...

 Trainable params: 11,173,962 (42.63 MB)

 Non-trainable params: 9,600 (37.50 KB)

INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Collective all_reduce tensors: 62 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1


I0000 00:00:1766462631.394627     123 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1766462631.423278     124 cuda_dnn.cc:529] Loaded cuDNN version 91002


INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
Epoch 1/100
INFO:tensorflow:Collective all_reduce tensors: 62 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, n

In [2]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import time
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import random

PROFILE = []

def tick(name, t0):
    PROFILE.append((name, time.perf_counter() - t0))

def now():
    return time.perf_counter()

t0 = now()

SEED = 12345
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

tf.config.threading.set_intra_op_parallelism_threads(0)
tf.config.threading.set_inter_op_parallelism_threads(0)

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

tick("environment_setup", t0)

print("TensorFlow:", tf.__version__)
print("GPUs:", gpus)
print("CPU threads:", os.cpu_count())

t0 = now()

(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
y_train = y_train.squeeze()
y_test = y_test.squeeze()

tick("dataset_load_and_preprocess", t0)

t0 = now()

VALIDATION_SPLIT = 0.1
n = x_train.shape[0]
idx = np.random.permutation(n)
val_n = int(n * VALIDATION_SPLIT)

x_val, y_val = x_train[idx[:val_n]], y_train[idx[:val_n]]
x_train, y_train = x_train[idx[val_n:]], y_train[idx[val_n:]]

tick("train_val_split", t0)

t0 = now()

BATCH_SIZE = 128

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(20000, seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

for _ in train_ds.take(5):
    pass

tick("tfdata_pipeline_build", t0)

print("Dataset ready")

def resnet_block(x, filters, stride=1):
    shortcut = x
    x = layers.Conv2D(filters, 3, strides=stride, padding="same", use_bias=False,
                      kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False,
                      kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, use_bias=False,
                                 kernel_regularizer=keras.regularizers.l2(1e-4))(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    x = layers.Add()([x, shortcut])
    x = layers.Activation("relu")(x)
    return x

def build_resnet18_model_parallel():
    inputs = keras.Input(shape=(32, 32, 3))
    with tf.device("/GPU:0"):
        x = layers.Conv2D(64, 3, padding="same", use_bias=False,
                          kernel_regularizer=keras.regularizers.l2(1e-4))(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = resnet_block(x, 64)
        x = resnet_block(x, 64)
        x = resnet_block(x, 128, stride=2)
    with tf.device("/GPU:1"):
        x = resnet_block(x, 128)
        x = resnet_block(x, 256, stride=2)
        x = resnet_block(x, 256)
        x = resnet_block(x, 512, stride=2)
        x = resnet_block(x, 512)
        x = layers.GlobalAveragePooling2D()(x)
        outputs = layers.Dense(10, activation="softmax")(x)
    return keras.Model(inputs, outputs)

t0 = now()

model = build_resnet18_model_parallel()

optimizer = keras.optimizers.SGD(
    learning_rate=0.1,
    momentum=0.9
)

model.compile(
    optimizer=optimizer,
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy()]
)

tick("model_build_and_compile", t0)

model.summary()

EPOCHS = 100

t0 = now()
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    verbose=1
)
tick("full_training_time", t0)

t0 = now()
model.evaluate(val_ds, verbose=0)
tick("validation_time", t0)

t0 = now()
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
tick("test_inference_time", t0)

print("Final Test Accuracy:", test_acc)

print("PROFILING SUMMARY")
for k, v in PROFILE:
    print(f"{k:30s}: {v:.4f} sec")

TensorFlow: 2.19.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
CPU threads: 4
Dataset ready


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_20 (Conv2D)  │ (None, 32, 32,    │      1,728 │ input_layer_1[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_20[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_17       │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_21 (Conv2D)  │ (None, 32, 32,    │     36,864 │ activation_17[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_21[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_18       │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_22 (Conv2D)  │ (None, 32, 32,    │     36,864 │ activation_18[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_22[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_8 (Add)         │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │ activation_17[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_19       │ (None, 32, 32,    │          0 │ add_8[0][0]       │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_23 (Conv2D)  │ (None, 32, 32,    │     36,864 │ activation_19[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_23[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_20       │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_24 (Conv2D)  │ (None, 32, 32,    │     36,864 │ activation_20[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_24[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_9 (Add)         │ (None, 32, 32,    │          0 │ batch_normalizat

 Total params: 11,183,562 (42.66 MB)

 Trainable params: 11,173,962 (42.63 MB)

 Non-trainable params: 9,600 (37.50 KB)

Epoch 1/100


InvalidArgumentError: Graph execution error:

Detected at node StatefulPartitionedCall defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>

  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance

  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start

  File "/usr/local/lib/python3.12/dist-packages/tornado/platform/asyncio.py", line 211, in start

  File "/usr/lib/python3.12/asyncio/base_events.py", line 645, in run_forever

  File "/usr/lib/python3.12/asyncio/base_events.py", line 1999, in _run_once

  File "/usr/lib/python3.12/asyncio/events.py", line 88, in _run

  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelbase.py", line 510, in dispatch_queue

  File "/usr/local/lib/python3.12/dist-packa